# The library's figures

Every figure in `docs/img/` is computed from this project's own code and data by `scripts/make_figures.py`.
This notebook runs each one and shows the result, grouped by library page. Change the code (a preset, the
tokenizer, a checkpoint), run the matching cell, and the figure follows.

Each section shows the figure as it is now in the repo; run its cell to regenerate it.

What each figure needs (it is skipped with a message if an input is missing):

| Input | Used by |
|---|---|
| `data/tokenizer.json` | run-1 tokenizer cuts, temperature |
| `data/pretrain/tokenizer.json`, `data/pretrain/manifest.csv` | run-2 cuts, corpus, network, attention flow |
| `showcase/run1/ckpt.pt` | temperature, run-1 attention map |
| `out/finetune/ckpt.pt` | temperature, network, attention flow, run-2 attention map |
| `showcase/run1/log.csv`, `showcase/run2_*_log.csv` | run 1 vs run 2 |


In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                      # run from the repo root
sys.path.insert(0, "scripts")
sys.path.insert(0, ".")

import make_figures as mf
from IPython.display import Image, display

FILES = {'books': 'fig_machado_books.png', 'bpe': 'fig_bpe_merges.png', 'cuts': 'fig_tokenizer_cuts.png', 'mask': 'fig_attention_mask.png', 'sqrtd': 'fig_sqrt_d.png', 'rope': 'fig_rope.png', 'act': 'fig_activations.png', 'kv': 'fig_kv_cache.png', 'ce': 'fig_cross_entropy.png', 'temp': 'fig_temperature.png', 'batch': 'fig_batch_shift.png', 'lr': 'fig_lr_schedule.png', 'corpus': 'fig_corpus.png', 'finetune': 'fig_finetune.png', 'run2': 'fig_run2.png', 'weights': 'fig_weights.png', 'network': 'fig_network.png', 'flow': 'fig_attention_flow.png'}

def make(key, width=820):
    """Regenerate one figure with make_figures.py, then show it."""
    mf.FIGURES[key]()
    path = os.path.join("docs", "img", FILES[key])
    if os.path.exists(path):
        display(Image(filename=path, width=width))

## 00 · The data

### Machado's 12 books by size

Counts the characters of each book in `data/corpus.txt`.

![Machado's 12 books by size](../docs/img/fig_machado_books.png)

In [ ]:
make("books")

## 01 · Tokenizer

### BPE merges on a toy corpus

Runs the real `get_stats` and `merge` from `tokenizer.py` for three rounds.

![BPE merges on a toy corpus](../docs/img/fig_bpe_merges.png)

In [ ]:
make("bpe")

### How run 1 and run 2 cut the same sentence

Encodes one sentence with both tokenizers (2,048 and 4,096 tokens).

![How run 1 and run 2 cut the same sentence](../docs/img/fig_tokenizer_cuts.png)

In [ ]:
make("cuts")

## 02 · Attention

### Scores, causal mask, softmax

A 5 × 5 example: raw scores, the future set to −∞, then the weights.

![Scores, causal mask, softmax](../docs/img/fig_attention_mask.png)

In [ ]:
make("mask")

### Why divide by √d

Attention weights with and without the scaling, on random vectors of dimension 64.

![Why divide by √d](../docs/img/fig_sqrt_d.png)

In [ ]:
make("sqrtd")

## 03 · Modern upgrades

### RoPE

Calls `precompute_rope` and `apply_rope`: rotation speeds, and a score that depends only on distance.

![RoPE](../docs/img/fig_rope.png)

In [ ]:
make("rope")

### KV cache with 6, 2 and 1 K/V heads

Memory needed to cache keys and values for a 256-token sequence.

![KV cache with 6, 2 and 1 K/V heads](../docs/img/fig_kv_cache.png)

In [ ]:
make("kv")

### ReLU vs SiLU, and the SwiGLU gate

The activation curves and a diagram of the gated feed-forward.

![ReLU vs SiLU, and the SwiGLU gate](../docs/img/fig_activations.png)

In [ ]:
make("act")

### Where the 11 million weights live

Every weight matrix of `TinyGPT`, drawn to scale from `named_parameters()`.

![Where the 11 million weights live](../docs/img/fig_weights.png)

In [ ]:
make("weights")

### Inside CausalSelfAttention.forward

Every tensor of the attention module with its shape, filled with the fine-tuned model's real values.

![Inside CausalSelfAttention.forward](../docs/img/fig_attention_flow.png)

In [ ]:
make("flow")

## 04 · Assembling the model

### Cross-entropy

Loss = −ln p for the probability given to the right token.

![Cross-entropy](../docs/img/fig_cross_entropy.png)

In [ ]:
make("ce")

### Temperature on a real prediction

Run 1 and run 2's top 10 next tokens after "Capitú olhou para", at three temperatures.

![Temperature on a real prediction](../docs/img/fig_temperature.png)

In [ ]:
make("temp")

### TinyGPT as a neural network

The same prompt through the model, with real attention weights, real feed-forward weights and real probabilities.

![TinyGPT as a neural network](../docs/img/fig_network.png)

In [ ]:
make("network")

## 05 · Training loop

### Input and target, shifted by one

One window from `get_batch`: each target is the next input token.

![Input and target, shifted by one](../docs/img/fig_batch_shift.png)

In [ ]:
make("batch")

### Learning-rate schedules

Calls `get_lr` with the real `mac` and `mac-pretrain` presets.

![Learning-rate schedules](../docs/img/fig_lr_schedule.png)

In [ ]:
make("lr")

## 06 · Training runs

### The run-2 corpus

Words by source and the largest authors, from `data/pretrain/manifest.csv`.

![The run-2 corpus](../docs/img/fig_corpus.png)

In [ ]:
make("corpus")

### Pretrain, then fine-tune

The two stages, and both learning-rate schedules on one log axis.

![Pretrain, then fine-tune](../docs/img/fig_finetune.png)

In [ ]:
make("finetune")

### Run 1 vs run 2

The loss curves of both runs and the loss per character on Machado's validation text.

![Run 1 vs run 2](../docs/img/fig_run2.png)

In [ ]:
make("run2")

## Attention heatmaps (`scripts/attention_maps.py`)

One heatmap per head of the last block, for a sentence of your choice. Row = the token looking, column = the
token looked at. Change the prompt or the block (`--layer 0` is the first block).

![Attention, run 2](../docs/img/attention_run2_block6.png)

In [ ]:
import subprocess
prompt = "Capitú olhou para mim e disse que não"
subprocess.run([sys.executable, "scripts/attention_maps.py",
                "--ckpt", "out/finetune/ckpt.pt", "--tokenizer", "data/pretrain/tokenizer.json",
                "--prompt", prompt, "--layer", "-1", "--out", "docs/img/attention_run2_block6.png"], check=True)
display(Image(filename="docs/img/attention_run2_block6.png", width=820))

## All at once

The same as `python scripts/make_figures.py` in a terminal: every figure whose inputs are present.

In [ ]:
for key in mf.FIGURES:
    print(key)
    mf.FIGURES[key]()